In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE

In [2]:
fs = pd.read_csv(r"..\..\data\pp\preprocessed_properties.csv")
fs

,property_type,society,sector,price_in_cr,total_area_sqft,bedrooms,bathrooms,balcony,floornum_category,study_room,...,others,facing,furnishing_type,age_possession,super_built_up_area_missing,built_up_area_missing,carpet_area_missing,plot_area_missing,area_per_bedroom,luxury_category
0,Independent Builder Floor,"sushant lok phase 1, gurgaon",sector 43,3.75,2701.0,4,4,2,Low-rise,0,...,1,south-east,Semi-Furnished,New Property,0,1,1,1,675.250000,Semi-Luxury
1,Independent House,Other,sector 48,0.75,1800.0,5,3,0,Low-rise,0,...,0,not available,Unfurnished,Moderately Old,1,0,1,1,360.000000,Budget
2,Independent House,tata primanti,sector 72,8.50,4500.0,4,4,3+,Low-rise,0,...,0,east,Unfurnished,Relatively New,1,1,1,0,1125.000000,Luxury
3,Flat,Other,sector 28,2.95,1370.0,3,3,1,Low-rise,0,...,0,not available,Unfurnished,Old Property,0,1,1,1,456.666667,Budget
4,Independent House,Other,sector 71,0.41,1044.0,3,2,2,Low-rise,0,...,0,north-west,Unfurnished,Relatively New,1,1,1,0,348.000000,Budget
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5424,Independent House,"sector 46, gurgaon",sector 46,5.15,1449.0,6,6,2,Low-rise,0,...,0,north-east,Unfurnished,Old Property,1,1,1,0,241.500000,Semi-Luxury
5425,Independent Builder Floor,dlf phase 1 ultra luxury builder floors,sector 26,8.75,5500.0,5,6,2,Low-rise,1,...,0,north,Semi-Furnished,Relatively New,1,1,0,1,1100.000000,Semi-Luxury
5426,Independent Builder Floor,dlf luxury builder floors,sector 25,5.55,3800.0,4,5,2,Low-rise,1,...,1,north-east,Semi-Furnished,New Property,1,1,0,1,950.000000,Luxury
5427,Independent Builder Floor,ansal api esencia,sector 67,2.61,2430.0,4,4,2,Low-rise,0,...,1,east,Semi-Furnished,Relatively New,0,1,1,1,607.500000,Luxury


In [3]:
fs.isnull().sum()

property_type                  0
society                        0
sector                         0
price_in_cr                    0
total_area_sqft                0
bedrooms                       0
bathrooms                      0
balcony                        0
floornum_category              0
study_room                     0
servant_room                   0
store_room                     0
pooja_room                     0
others                         0
facing                         0
furnishing_type                0
age_possession                 0
super_built_up_area_missing    0
built_up_area_missing          0
carpet_area_missing            0
plot_area_missing              0
area_per_bedroom               0
luxury_category                0
dtype: int64

In [4]:
corr = fs.corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=True,          
    color_continuous_scale='RdBu_r',  
    title="Correlation Heatmap"
)

fig.update_layout(
    xaxis_title="Features",
    yaxis_title="Features",
    width=800,
    height=700
)

In [5]:
corr['price_in_cr'].sort_values(ascending= False)

price_in_cr                    1.000000
total_area_sqft                0.722661
bathrooms                      0.651182
bedrooms                       0.606585
area_per_bedroom               0.410975
servant_room                   0.360146
super_built_up_area_missing    0.358329
store_room                     0.325850
pooja_room                     0.320846
study_room                     0.305801
others                         0.026463
carpet_area_missing            0.013024
built_up_area_missing         -0.064983
plot_area_missing             -0.491028
Name: price_in_cr, dtype: float64

In [6]:
df = fs.copy()

###  Step 1: Separate Features and Target

In [7]:
X = df.drop(columns=["price_in_cr"])
y = np.log1p(df["price_in_cr"])   


print("Full dataset shape:", X.shape)

Full dataset shape: (5429, 22)


##### Split FIRST

In [8]:
X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape used for feature ranking:", X_train.shape)

Training shape used for feature ranking: (4343, 22)


### Step 2: Encode Categorical Features (For Tree Models)

In [9]:
X_train_enc = X_train.copy()
y_train_enc = np.log1p(y_train)

categorical_cols = X_train_enc.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:", categorical_cols)

if len(categorical_cols) > 0:
    encoder = OrdinalEncoder()
    X_train_enc[categorical_cols] = encoder.fit_transform(
        X_train_enc[categorical_cols]
    )

print("Encoded training shape:", X_train_enc.shape)

Categorical columns: ['property_type', 'society', 'sector', 'balcony', 'floornum_category', 'facing', 'furnishing_type', 'age_possession', 'luxury_category']
Encoded training shape: (4343, 22)


### Step 3: Random Forest Feature Importance

In [10]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_enc, y_train_enc)

rf_imp = pd.Series(
    rf.feature_importances_,
    index=X_train_enc.columns,
    name="rf"
)

rf_imp.sort_values(ascending=False).head(20)

total_area_sqft          0.615557
property_type            0.148945
sector                   0.074655
plot_area_missing        0.034058
area_per_bedroom         0.025159
society                  0.023649
bathrooms                0.016273
balcony                  0.010021
servant_room             0.007776
facing                   0.007383
bedrooms                 0.007156
luxury_category          0.006691
age_possession           0.005428
furnishing_type          0.004912
pooja_room               0.002334
floornum_category        0.002119
built_up_area_missing    0.001584
study_room               0.001547
carpet_area_missing      0.001522
store_room               0.001357
Name: rf, dtype: float64

### Step 4: Gradient Boosting Importance

In [11]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_enc, y_train_enc)

gb_imp = pd.Series(
    gb.feature_importances_,
    index=X_train_enc.columns,
    name="gb"
)

gb_imp.sort_values(ascending=False).head()

total_area_sqft      0.590894
property_type        0.139054
bathrooms            0.075023
sector               0.074919
plot_area_missing    0.038228
Name: gb, dtype: float64

###  Step 5: Permutation Importance

In [12]:
perm = permutation_importance(
    rf,
    X_train_enc,
    y_train_enc,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_imp = pd.Series(
    perm.importances_mean,
    index=X_train_enc.columns,
    name="perm"
)

perm_imp.sort_values(ascending=False).head()

total_area_sqft      0.923557
property_type        0.201608
sector               0.166678
plot_area_missing    0.075878
bathrooms            0.039393
Name: perm, dtype: float64

### Step 6: REF

In [13]:
rfe = RFE(
    RandomForestRegressor(n_estimators=200, random_state=42),
    n_features_to_select= 16
)

rfe.fit(X_train_enc, y_train_enc)

rfe_rank = pd.Series(
    rfe.ranking_,
    index=X_train_enc.columns,
    name="rfe_rank"
)

rfe_rank.sort_values().head(20)

property_type            1
society                  1
sector                   1
total_area_sqft          1
bedrooms                 1
bathrooms                1
balcony                  1
floornum_category        1
servant_room             1
pooja_room               1
furnishing_type          1
facing                   1
age_possession           1
luxury_category          1
area_per_bedroom         1
plot_area_missing        1
carpet_area_missing      2
built_up_area_missing    3
study_room               4
store_room               5
Name: rfe_rank, dtype: int64

### Step 7: Combine All Importance Scores

In [14]:
importance_df = pd.concat([rf_imp, gb_imp, perm_imp], axis=1)

# Normalize columns
importance_df = (importance_df - importance_df.min()) / (
    importance_df.max() - importance_df.min()
)

importance_df["avg_importance"] = importance_df.mean(axis=1)

# Convert RFE ranking into score
importance_df["rfe_score"] = 1 / rfe_rank

importance_df["final_score"] = (
    importance_df["avg_importance"] + importance_df["rfe_score"]
) / 2

importance_df = importance_df.sort_values("final_score", ascending=False)

importance_df.head(20)

,rf,gb,perm,avg_importance,rfe_score,final_score
total_area_sqft,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
property_type,0.241281,0.235327,0.217880,0.231496,1.000000,0.615748
sector,0.120485,0.126789,0.180039,0.142438,1.000000,0.571219
plot_area_missing,0.054473,0.064695,0.081672,0.066947,1.000000,0.533473
bathrooms,0.025554,0.126965,0.042146,0.064888,1.000000,0.532444
society,0.037548,0.027793,0.036383,0.033908,1.000000,0.516954
area_per_bedroom,0.040004,0.020059,0.039147,0.033070,1.000000,0.516535
servant_room,0.011739,0.024034,0.019229,0.018334,1.000000,0.509167
bedrooms,0.010731,0.031176,0.012629,0.018178,1.000000,0.509089
balcony,0.015388,0.009421,0.012874,0.012561,1.000000,0.506281


### Step 8: Final Top 16 

In [15]:
selected_features = importance_df.head(15).index.tolist()

print("Selected Top Features:")
for f in selected_features:
    print("-", f)

Selected Top Features:
- total_area_sqft
- property_type
- sector
- plot_area_missing
- bathrooms
- society
- area_per_bedroom
- servant_room
- bedrooms
- balcony
- luxury_category
- furnishing_type
- facing
- age_possession
- floornum_category


In [16]:
fs_df = df[selected_features + ['price_in_cr']].copy()

print("Final shape after feature selection:", fs_df.shape)
fs_df.head()

Final shape after feature selection: (5429, 16)


,total_area_sqft,property_type,sector,plot_area_missing,bathrooms,society,area_per_bedroom,servant_room,bedrooms,balcony,luxury_category,furnishing_type,facing,age_possession,floornum_category,price_in_cr
0,2701.0,Independent Builder Floor,sector 43,1,4,"sushant lok phase 1, gurgaon",675.250000,1,4,2,Semi-Luxury,Semi-Furnished,south-east,New Property,Low-rise,3.75
1,1800.0,Independent House,sector 48,1,3,Other,360.000000,0,5,0,Budget,Unfurnished,not available,Moderately Old,Low-rise,0.75
2,4500.0,Independent House,sector 72,0,4,tata primanti,1125.000000,1,4,3+,Luxury,Unfurnished,east,Relatively New,Low-rise,8.50
3,1370.0,Flat,sector 28,1,3,Other,456.666667,0,3,1,Budget,Unfurnished,not available,Old Property,Low-rise,2.95
4,1044.0,Independent House,sector 71,0,2,Other,348.000000,0,3,2,Budget,Unfurnished,north-west,Relatively New,Low-rise,0.41
